# **Long Short Term Memory**
Long Short-Term Memory (LSTM) is an enhanced version of the Recurrent Neural Network (RNN) designed by Hochreiter and Schmidhuber. LSTMs can capture long-term dependencies in sequential data making them ideal for tasks like language translation, speech recognition and time series forecasting. Unlike traditional RNNs which use a single hidden state passed through time LSTMs introduce a memory cell that holds information over extended periods addressing the challenge of learning long-term dependencies.

![](https://media.geeksforgeeks.org/wp-content/uploads/20250528172141936296/architecture_of_lstms.webp)

## Major Problems with RNN (That LSTM solves)

- ### **Vanishing Gradient Problem**
    This is the most significant problem LSTMs solve. In standard RNNs, as information is backpropagated through many time steps, the gradients can become extremely small, approaching zero. This makes it impossible for the network to update its weights effectively for earlier time steps, effectively stopping the learning process.
    
- ### **Long-Term Dependency Problem**
    Due to vanishing gradients, standard RNNs "forget" information from distant past time steps. LSTMs use a specialized cell state and gating mechanism to selectively remember important information over long periods, allowing them to bridge large gaps in sequences.

## Architecture of LSTM

![](https://media.geeksforgeeks.org/wp-content/uploads/20250404172141987003/gate_of_lstm.webp)

### 1. **Forget Gate**
The information that is no longer useful in the cell state is removed with the forget gate. Two inputs $x_{t}$ (input at the particular time) and $h_{t-1}$ (previous cell output) are fed to the gate and multiplied with weight matrices followed by the addition of bias. The resultant is passed through sigmoid activation function which gives output in range of [0,1]. If for a particular cell state the output is 0 or near to 0, the piece of information is forgotten and for output of 1 or near to 1, the information is retained for future use.

Equation used in this gate:

$$ f_{t} = σ(W_{f} ⋅ [h_{t-1}, x_{t}] + b_{f}) $$

### 2. **Input Gate**
The addition of useful information to the cell state is done by the input gate. First the information is regulated using the sigmoid function and filter the values to be remembered similar to the forget gate using inputs $h_{t-1}$ and $x_{t}$. Then, a vector is created using tanh function that gives an output from -1 to +1 which contains all the possible values from $h_{t-1}$ and $X_{t}$. At last the values of the vector and the regulated values are multiplied to obtain the useful information.

The equation for the input gate is:

$$ i_{t} = σ(W_{i} ⋅ [h_{t-1}, x_{t}] + b_{i}) $$
$$ ĉ_{t} = tanh(W_{c} ⋅ [h_{t-1}, x_{t}] + b_{c}) $$

We multiply the previous state by $f_{t}$ effectively filtering out the information we had decided to ignore earlier. Then we add $i_{t} ⊙ c_{t}$  which represents the new candidate values scaled by how much we decided to update each state value.

## Implementation

In [2]:
import nltk
nltk.download('gutenberg')

from nltk.corpus import gutenberg
data = gutenberg.raw('shakespeare-hamlet.txt')

with open('hamlet.txt', 'w') as file:
    file.write(data)

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

with open('hamlet.txt', 'r') as file:
    text = file.read().lower()

tokeniser = Tokenizer()
tokeniser.fit_on_texts([text])
word_count = len(tokeniser.word_index)+1
print(f"There are {word_count} words.")

There are 4818 words.


In [4]:
lines = []

for line in text.split('\n'):
    lines.append(tokeniser.texts_to_sequences([line])[0])

lines

[[1, 687, 4, 45, 41, 1886, 1887, 1888],
 [],
 [],
 [1180, 1889, 1890, 1891],
 [],
 [57, 407, 2, 1181, 177, 1892],
 [],
 [407, 1182, 63],
 [408, 162, 377, 21, 247, 882],
 [18, 66],
 [],
 [451, 224, 248, 1, 30],
 [],
 [408, 407],
 [451, 25],
 [],
 [408, 6, 43, 62, 1893, 96, 18, 566],
 [],
 [451, 71, 51, 1894, 567, 378, 80, 3, 273, 1181],
 [],
 [408, 20, 16, 1895, 114, 379, 71, 883, 491],
 [2, 5, 92, 688, 58, 144],
 [],
 [346, 29, 6, 108, 568, 884],
 [408, 14, 7, 885, 1896],
 [],
 [346, 64, 380, 37, 6, 47, 689, 120, 2],
 [347, 1, 1897, 4, 8, 257, 492, 68, 87, 149],
 [57, 120, 2, 347],
 [],
 [408, 5, 117, 5, 139, 68, 247, 1182, 63],
 [48, 205, 3, 16, 409],
 [],
 [140, 2, 1898, 348, 3, 1, 493],
 [],
 [408, 79, 6, 46, 124],
 [],
 [140, 125, 1899, 322, 1183, 121, 83, 1900, 6],
 [1901, 407, 258, 8, 494, 79, 6, 380],
 [],
 [218, 408],
 [],
 [140, 1902, 407],
 [],
 [451, 94, 24, 13, 120, 63],
 [48, 7, 495, 4, 28],
 [],
 [451, 236, 120, 236, 46, 347],
 [],
 [140, 24, 258, 16, 167, 886, 131, 3, 12

In [5]:
# n-gram sequence
input_seq = []

for line in lines:
    for i in range(1, len(line)):
        input_seq.append(line[:i+1])

input_seq

[[1, 687],
 [1, 687, 4],
 [1, 687, 4, 45],
 [1, 687, 4, 45, 41],
 [1, 687, 4, 45, 41, 1886],
 [1, 687, 4, 45, 41, 1886, 1887],
 [1, 687, 4, 45, 41, 1886, 1887, 1888],
 [1180, 1889],
 [1180, 1889, 1890],
 [1180, 1889, 1890, 1891],
 [57, 407],
 [57, 407, 2],
 [57, 407, 2, 1181],
 [57, 407, 2, 1181, 177],
 [57, 407, 2, 1181, 177, 1892],
 [407, 1182],
 [407, 1182, 63],
 [408, 162],
 [408, 162, 377],
 [408, 162, 377, 21],
 [408, 162, 377, 21, 247],
 [408, 162, 377, 21, 247, 882],
 [18, 66],
 [451, 224],
 [451, 224, 248],
 [451, 224, 248, 1],
 [451, 224, 248, 1, 30],
 [408, 407],
 [451, 25],
 [408, 6],
 [408, 6, 43],
 [408, 6, 43, 62],
 [408, 6, 43, 62, 1893],
 [408, 6, 43, 62, 1893, 96],
 [408, 6, 43, 62, 1893, 96, 18],
 [408, 6, 43, 62, 1893, 96, 18, 566],
 [451, 71],
 [451, 71, 51],
 [451, 71, 51, 1894],
 [451, 71, 51, 1894, 567],
 [451, 71, 51, 1894, 567, 378],
 [451, 71, 51, 1894, 567, 378, 80],
 [451, 71, 51, 1894, 567, 378, 80, 3],
 [451, 71, 51, 1894, 567, 378, 80, 3, 273],
 [451, 71

In [6]:
import numpy as np

max_len = max([len(x) for x in input_seq])
input_seq = np.array(pad_sequences(input_seq, maxlen=max_len, padding='pre'))
input_seq

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]], dtype=int32)

In [7]:
# X, y split

X, y = input_seq[:, :-1], input_seq[:, -1]
X, y

(array([[   0,    0,    0, ...,    0,    0,    1],
        [   0,    0,    0, ...,    0,    1,  687],
        [   0,    0,    0, ...,    1,  687,    4],
        ...,
        [   0,    0,    0, ...,  687,    4,   45],
        [   0,    0,    0, ...,    4,   45, 1047],
        [   0,    0,    0, ...,   45, 1047,    4]], dtype=int32),
 array([ 687,    4,   45, ..., 1047,    4,  193], dtype=int32))

In [8]:
# converting Y into categories

y = tf.keras.utils.to_categorical(y, num_classes=word_count)
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential()
model.add(Embedding(word_count, 100, input_length=max_len))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(word_count, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [30]:
history = model.fit(X_train, y_train, epochs=50, validation_data=(X_test, y_test))

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.0287 - loss: 7.1383 - val_accuracy: 0.0338 - val_loss: 6.7027
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.0394 - loss: 6.4632 - val_accuracy: 0.0420 - val_loss: 6.7592
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.0436 - loss: 6.3212 - val_accuracy: 0.0488 - val_loss: 6.7860
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.0507 - loss: 6.1596 - val_accuracy: 0.0482 - val_loss: 6.8018
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.0542 - loss: 6.0073 - val_accuracy: 0.0575 - val_loss: 6.7975
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.0632 - loss: 5.8791 - val_accuracy: 0.0606 - val_loss: 6.8609
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.0729 - loss: 5.6979 - val_accuracy: 0.0628 - val_loss: 6.9036
Epoch 8/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.0776 - loss: 5.5965 - val_accur

Here the accuracy just went to 43% as hamlet is a complex dataset that needed  ore transformation and optimisation. Also there is high overfitting.\
Howevever this practical was just to get a hands-on.

# **Gated Recurrent Units**
Gated Recurrent Units (GRUs) are a type of RNN architecture that use gating mechanisms—specifically a reset gate and an update gate—to manage information flow, solving the vanishing gradient problem. GRUs are faster to train and more efficient than LSTMs, making them ideal for sequence tasks like language modeling, speech recognition, and time-series forecasting.

![](https://media.geeksforgeeks.org/wp-content/uploads/20250106111020535923/GRU.webp)

## Components

### 1. **Update Gate**
This gate decides how much of the information from the past should be kept for future steps. It helps the GRU remember important details.

$$ z_{t} = σ(W_{2} ⋅ [h_{t-1}, x_{t}]) $$

### 2. **Reset Gate**
This gate decides how much past information should be forgotten. If something is no longer relevant it forget that part.

$$ r_{t} = σ(W_{z} ⋅ [h_{t-1}, x_{t}]) $$

### 3. **Candidate hidden state**
This is the potential new hidden state calculated based on the current input and the previous hidden state.

$$ h'_{t} = tanh(W_{h} ⋅ [r_{t} ⋅ h_{t-1}, x_{t}]) $$

### 4. **Hidden state**
The final hidden state is a weighted average of the previous hidden state $h_{t-1}$ the candidate hidden state $h'_{t}$ based on the update gate $z_{t}$.

$$ h_{t} = (1 - z_{t}) ⋅ h_{t-1} + z_{t} ⋅ h'_{t} $$